# Paso 4.4 — Comparación ML clásico vs Deep Learning
## Sub-paso 4.4.C: Síntesis transversal y guía decisional

**Desafío Profesional de Data Science — Digital House**  
Proyecto: Cambio Climático y Energías Renovables en Argentina

---

### 🎯 ¿Qué hacemos en este sub-paso?

Los sub-pasos 4.4.A y 4.4.B nos dejaron dos tablas con los siete modelos del proyecto. Cada tabla responde una pregunta concreta — quién predice mejor en regresión global y quién forecastea mejor las renovables argentinas. Ahora cerramos el 4.4 con una pregunta de orden superior:

> **¿Cuándo conviene cada enfoque?**

Esa pregunta no se contesta con una métrica. Se contesta entendiendo los **trade-offs no-métricos** —tiempo, parámetros, interpretabilidad, robustez, mantenimiento— y cómo se cruzan con la naturaleza del problema. Eso es lo que construimos acá.

### 🗺️ Estructura del notebook

1. **Setup**: imports y carga de los resultados consolidados de 4.4.A y 4.4.B.
2. **Tabla transversal — 7 modelos × 6 dimensiones**: la síntesis principal.
3. **Visualizaciones de la síntesis**: heatmap normalizado y gráfico de Pareto Performance vs Costo.
4. **Guía decisional**: árbol de decisión visual.
5. **Storytelling de cierre**: las grandes lecciones del 4.4.
6. **Próximo paso**: link al 4.5.

### ⚙️ Por qué no reentrenamos los modelos en este notebook

Los siete modelos ya fueron entrenados en 4.4.A y 4.4.B con seeds y splits correctos. Re-entrenarlos acá produciría números levemente distintos por la varianza inter-seed —una verdad incómoda que vimos en el 4.4.B—, y eso *contradiría* la narrativa: la tabla de síntesis no debería tener números distintos a los de A y B. Por eso, lo que hacemos acá es **consolidar los resultados medidos**, no re-medirlos. Si querés re-ejecutar todo desde cero, las corridas reproducibles están en los notebooks 4.4.A y 4.4.B.

> ✅ **Notebook ejecutado**
>
> Los números de la tabla de síntesis vienen de las corridas reales de 4.4.A y 4.4.B. Para re-ejecutar este notebook en Colab, abrilo y corré *Runtime → Run all* — no requiere datasets externos.

---

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import warnings
warnings.filterwarnings('ignore')

print('Setup completo.')

### 1.1 Resultados consolidados de 4.4.A y 4.4.B

Cargamos como constantes los números obtenidos en las corridas anteriores. Para los modelos con múltiples seeds (MLP del 4.1 y LSTM Tuneada del 4.3), guardamos también el desvío inter-seed.

In [ ]:
# Resultados de 4.4.A — Bloque de regresión global (target: co2_per_capita)
resultados_A = [
    {'Modelo': 'Regresión Múltiple', 'Bloque': 'Regresión',
     'Métrica principal': 'R²', 'Valor': 0.7583, 'Desvío seeds': None,
     'Tiempo (s)': 0.002, '# Params': 7},
    {'Modelo': 'Ridge (α=10)',       'Bloque': 'Regresión',
     'Métrica principal': 'R²', 'Valor': 0.7567, 'Desvío seeds': None,
     'Tiempo (s)': 0.002, '# Params': 7},
    {'Modelo': 'Lasso (α=0.01)',     'Bloque': 'Regresión',
     'Métrica principal': 'R²', 'Valor': 0.7586, 'Desvío seeds': None,
     'Tiempo (s)': 0.002, '# Params': 7},
    {'Modelo': 'MLP Medio',          'Bloque': 'Regresión',
     'Métrica principal': 'R²', 'Valor': 0.9309, 'Desvío seeds': 0.0177,
     'Tiempo (s)': 33.7,  '# Params': 9985},
]

# Resultados de 4.4.B — Bloque de series temporales (target: gen_renovable_total_GWh)
resultados_B = [
    {'Modelo': 'ARIMA(1,1,1)',          'Bloque': 'Forecasting',
     'Métrica principal': 'MAPE', 'Valor': 26.68, 'Desvío seeds': None,
     'Tiempo (s)': 0.033, '# Params': 3},
    {'Modelo': 'LSTM Simple (4.2)',     'Bloque': 'Forecasting',
     'Métrica principal': 'MAPE', 'Valor': 20.61, 'Desvío seeds': None,
     'Tiempo (s)': 9.9,   '# Params': 4385},
    {'Modelo': 'LSTM Tuneada (4.3)',    'Bloque': 'Forecasting',
     'Métrica principal': 'MAPE', 'Valor': 20.70, 'Desvío seeds': 3.47,
     'Tiempo (s)': 11.0,  '# Params': 4385},
]

df_consolidado = pd.DataFrame(resultados_A + resultados_B)
df_consolidado

---

## 2. La tabla transversal — 7 modelos × 6 dimensiones

Definimos las **6 dimensiones de comparación** que trascienden la métrica de cada problema:

| # | Dimensión | Tipo | Cómo la calculamos |
|---|---|---|---|
| 1 | **Performance relativa** | Cuantitativa | Ranking dentro del bloque (1 = mejor de su categoría) |
| 2 | **Tiempo de entrenamiento** | Cuantitativa | Segundos cronometrados en 4.4.A y 4.4.B |
| 3 | **# Parámetros** | Cuantitativa | `count_params()` o # coeficientes |
| 4 | **Interpretabilidad** | Cualitativa | Alta / Media / Baja |
| 5 | **Robustez inter-seed** | Cuantitativa o "—" | Desvío en la métrica; "—" para determinísticos |
| 6 | **Mantenimiento producción** | Cualitativa | Bajo / Medio / Alto costo |

**Por qué "performance relativa" y no absoluta**: mezclar R² (regresión) con MAPE (forecasting) en una misma columna sería engañoso —son métricas con dirección y escala distintas—. El ranking dentro del bloque conserva la información de "quién ganó" sin mezclar peras con manzanas.

In [ ]:
# Calcular ranking dentro de cada bloque
def asignar_ranking(df):
    df = df.copy()
    # En regresión, mayor R² = mejor → ranking ascendente por -R²
    # En forecasting, menor MAPE = mejor → ranking ascendente por MAPE
    rankings = []
    for bloque in df['Bloque'].unique():
        sub = df[df['Bloque'] == bloque].copy()
        if bloque == 'Regresión':
            sub['Ranking'] = sub['Valor'].rank(ascending=False, method='min').astype(int)
        else:
            sub['Ranking'] = sub['Valor'].rank(ascending=True, method='min').astype(int)
        rankings.append(sub)
    return pd.concat(rankings).sort_index()

df_consolidado = asignar_ranking(df_consolidado)

# Anotaciones cualitativas — fundamentadas en lo que aprendimos en cada paso
interpretabilidad = {
    'Regresión Múltiple': 'Alta',  # coeficientes directos
    'Ridge (α=10)':       'Alta',
    'Lasso (α=0.01)':     'Alta',  # además hace selección de features
    'MLP Medio':          'Baja',  # caja negra; SHAP/LIME ayudan pero son post-hoc
    'ARIMA(1,1,1)':       'Media', # parámetros AR/MA tienen lectura, pero menos directa
    'LSTM Simple (4.2)':  'Baja',
    'LSTM Tuneada (4.3)': 'Baja',
}

mantenimiento = {
    'Regresión Múltiple': 'Bajo',  # entrenar es trivial, sin hiperparámetros
    'Ridge (α=10)':       'Bajo',  # un único hiperparámetro α
    'Lasso (α=0.01)':     'Bajo',
    'MLP Medio':          'Medio', # arquitectura, lr, batch, regularización, seeds…
    'ARIMA(1,1,1)':       'Bajo',  # selección de orden p,d,q es estándar
    'LSTM Simple (4.2)':  'Medio',
    'LSTM Tuneada (4.3)': 'Alto',  # tuning fue caro y devolvió poco
}

df_consolidado['Interpretabilidad'] = df_consolidado['Modelo'].map(interpretabilidad)
df_consolidado['Mantenimiento']     = df_consolidado['Modelo'].map(mantenimiento)

# Formatear desvío seeds como string — usar pd.isna para manejar el caso None→NaN
def fmt_seed(row):
    if pd.isna(row['Desvío seeds']):
        return '—'
    if row['Bloque'] == 'Regresión':
        return f"±{row['Desvío seeds']:.3f}"
    else:
        return f"±{row['Desvío seeds']:.2f}"

df_consolidado['Robustez (σ inter-seed)'] = df_consolidado.apply(fmt_seed, axis=1)

# Ordenar columnas para la presentación final
tabla_final = df_consolidado[[
    'Modelo', 'Bloque', 'Ranking', 'Métrica principal', 'Valor',
    'Tiempo (s)', '# Params',
    'Interpretabilidad', 'Robustez (σ inter-seed)', 'Mantenimiento',
]].copy()
tabla_final = tabla_final.rename(columns={'Valor': 'Valor métrica'})
tabla_final

### 2.1 Lectura por dimensión

**Performance relativa**: en regresión, el MLP es el #1 por amplio margen. En forecasting, la LSTM Simple y la Tuneada empatan empíricamente (la Simple gana por 0.09 puntos de MAPE, dentro del ruido); ARIMA queda último.

**Tiempo y parámetros**: tres órdenes de magnitud separan a los lineales (`<0.01s, ~7 params`) de las redes (`~10–34s, ~4k–10k params`). En producción esto importa: re-entrenar Ridge en una pipeline diaria es trivial; re-entrenar un MLP con cada ingesta requiere un orquestador serio.

**Interpretabilidad**: los lineales ganan trivialmente —los coeficientes son la explicación—. ARIMA está en el medio (los parámetros AR/MA tienen lectura, aunque menos directa). Las redes neuronales son cajas negras y dependen de técnicas post-hoc (SHAP, LIME, attention maps) para explicar predicciones individuales.

**Robustez inter-seed**: los modelos determinísticos (`—`) son perfectamente reproducibles —siempre dan el mismo resultado—. Las redes tienen variabilidad: el MLP es estable (±0.018 en R², muy chico comparado con su métrica de 0.93), pero la LSTM Tuneada es **bastante inestable** (±3.47 en MAPE, comparable al margen entre modelos del bloque). Esto se vuelve un argumento de peso en producción.

**Mantenimiento**: los lineales se entrenan en milisegundos sin hiperparámetros relevantes, así que son bajo costo de mantenimiento. Las redes requieren cuidado: arquitectura, learning rate, batch size, regularización, seeds. Marcamos a la LSTM Tuneada como *alto* costo porque su tuning fue caro y devolvió poco —en producción, re-tunear cada vez que cambia la distribución de datos es un costo real.

---

## 3. Visualizaciones de la síntesis

### 3.1 Heatmap normalizado — performance, costo y robustez

Para visualizar los trade-offs en un solo gráfico, normalizamos las dimensiones cuantitativas a una escala 0–1 donde **1 = mejor**:

- **Performance**: dentro del bloque, normalizamos contra el mejor del bloque.
- **Velocidad** (1 / tiempo): el más rápido tiene 1.
- **Compacidad** (1 / # params): el más chico tiene 1.
- **Estabilidad**: los determinísticos tienen 1; los estocásticos se penalizan según su desvío relativo.

Esto permite ver de un vistazo en qué dimensión brilla cada modelo.

In [ ]:
df_norm = df_consolidado.copy().reset_index(drop=True)

# 1) Performance normalizada dentro del bloque
def normalizar_performance(row):
    sub = df_norm[df_norm['Bloque'] == row['Bloque']]
    if row['Bloque'] == 'Regresión':
        # mayor R² = mejor → normalizar contra el máximo
        return row['Valor'] / sub['Valor'].max()
    else:
        # menor MAPE = mejor → invertir
        return sub['Valor'].min() / row['Valor']

df_norm['Performance'] = df_norm.apply(normalizar_performance, axis=1)

# 2) Velocidad: 1/tiempo, normalizado al máximo global
df_norm['Velocidad'] = 1.0 / df_norm['Tiempo (s)']
df_norm['Velocidad'] = df_norm['Velocidad'] / df_norm['Velocidad'].max()

# 3) Compacidad: 1/n_params, escala log para que no aplaste todo
df_norm['Compacidad'] = 1.0 / np.log10(df_norm['# Params'] + 1)
df_norm['Compacidad'] = df_norm['Compacidad'] / df_norm['Compacidad'].max()

# 4) Estabilidad: los determinísticos tienen 1 (perfectamente reproducibles);
#    los estocásticos se penalizan según su desvío relativo
def estabilidad(row):
    if pd.isna(row['Desvío seeds']):
        return 1.0
    desvio_relativo = row['Desvío seeds'] / abs(row['Valor'])
    return max(0.0, 1.0 - desvio_relativo)

df_norm['Estabilidad'] = df_norm.apply(estabilidad, axis=1)

# Construir la matriz para el heatmap
matriz = df_norm.set_index('Modelo')[['Performance', 'Velocidad', 'Compacidad', 'Estabilidad']]
matriz_disp = matriz.round(2)
matriz_disp

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))

# Colormap: rojo claro (peor) → verde oscuro (mejor)
cmap = LinearSegmentedColormap.from_list(
    'RdYlGn_custom', ['#fceaea', '#fff4d6', '#d9eed1', '#5b9b58'], N=256
)

im = ax.imshow(matriz.values, cmap=cmap, vmin=0, vmax=1, aspect='auto')

# Etiquetas
ax.set_xticks(range(len(matriz.columns)))
ax.set_xticklabels(matriz.columns, fontsize=11)
ax.set_yticks(range(len(matriz.index)))
ax.set_yticklabels(matriz.index, fontsize=10)

# Anotar cada celda con el valor
for i in range(matriz.shape[0]):
    for j in range(matriz.shape[1]):
        v = matriz.values[i, j]
        color_text = 'white' if v > 0.55 else '#222'
        ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                color=color_text, fontsize=10, fontweight='bold')

# Línea separadora entre bloques
n_reg = (df_consolidado['Bloque'] == 'Regresión').sum()
ax.axhline(n_reg - 0.5, color='black', linewidth=1.5)

# Colorbar
cbar = plt.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
cbar.set_label('Score normalizado (1 = mejor en su dimensión)', fontsize=10)

ax.set_title('Síntesis transversal — 7 modelos × 4 dimensiones cuantitativas', fontsize=13, pad=12)
plt.tight_layout()
plt.show()

**Cómo leer este heatmap.** Las dos zonas más claras del mapa son los modelos lineales (arriba), que dominan en velocidad, compacidad y estabilidad pero pierden en performance, y las redes neuronales (centro y abajo), que ganan en performance pero pierden en todo lo demás. **Esa diagonal de colores es exactamente el trade-off central de Machine Learning.** No hay modelo "verde en todo": cada uno hace concesiones.

### 3.2 Pareto: Performance vs Costo

Otra forma de ver lo mismo: graficamos cada modelo en un plano de **performance** (eje vertical) vs **costo computacional** (eje horizontal, en escala logarítmica de # de parámetros). Los modelos en la "frontera de Pareto" son los que no están dominados por ningún otro —ninguna otra opción les gana en ambas dimensiones a la vez—.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.2))

# === Panel izquierdo: Regresión ===
df_reg = df_norm[df_norm['Bloque'] == 'Regresión']
ax1.scatter(df_reg['# Params'], df_reg['Valor'],
            s=180, c='#7F77DD', edgecolor='black', linewidth=1, alpha=0.85, zorder=3)
for _, row in df_reg.iterrows():
    ax1.annotate(row['Modelo'],
                 (row['# Params'], row['Valor']),
                 xytext=(8, 6), textcoords='offset points', fontsize=10)
ax1.set_xscale('log')
ax1.set_xlabel('# Parámetros (escala log)', fontsize=11)
ax1.set_ylabel('R² en test', fontsize=11)
ax1.set_title('Bloque A — Regresión global', fontsize=12)
ax1.grid(alpha=0.3, zorder=0)
ax1.set_ylim(0.65, 1.0)

# === Panel derecho: Forecasting ===
df_fcst = df_norm[df_norm['Bloque'] == 'Forecasting']
ax2.scatter(df_fcst['# Params'], df_fcst['Valor'],
            s=180, c='#1D9E75', edgecolor='black', linewidth=1, alpha=0.85, zorder=3)
# Errorbar inter-seed donde corresponda
for _, row in df_fcst.iterrows():
    if row['Desvío seeds'] is not None:
        ax2.errorbar(row['# Params'], row['Valor'],
                     yerr=row['Desvío seeds'], fmt='none',
                     color='black', capsize=5, alpha=0.6, zorder=2)
    ax2.annotate(row['Modelo'],
                 (row['# Params'], row['Valor']),
                 xytext=(8, 6), textcoords='offset points', fontsize=10)
ax2.set_xscale('log')
ax2.set_xlabel('# Parámetros (escala log)', fontsize=11)
ax2.set_ylabel('MAPE en test (%, menor = mejor)', fontsize=11)
ax2.invert_yaxis()  # invertimos para que "arriba" siga siendo "mejor"
ax2.set_title('Bloque B — Forecasting Argentina', fontsize=12)
ax2.grid(alpha=0.3, zorder=0)

plt.suptitle('Pareto: Performance vs Costo (en parámetros)',
             fontsize=13, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

**Lo que dice este Pareto.**

- **En regresión** (panel izquierdo) la frontera de Pareto está formada por solo dos puntos: **Lasso** (barato y razonable) y **MLP** (caro pero claramente superior). Los otros lineales (Ridge, Regresión Múltiple) están dominados por Lasso —dan lo mismo pero no aportan nada nuevo—. Si pasás de 7 a ~10.000 parámetros, ganás 17 puntos de R². Ese precio puede valer la pena.
- **En forecasting** (panel derecho) la frontera incluye a **ARIMA** y a la **LSTM Simple**. La barra de error vertical sobre la LSTM Tuneada muestra justo lo que conviene ver: el rango de la Tuneada solapa con el de la Simple, así que la Tuneada **no expande la frontera** de Pareto. Pasar de 3 a 4.385 parámetros sí compra 6 puntos de MAPE; tunear ese mismo modelo no compra nada significativo.

---

## 4. Guía decisional — ¿cuándo conviene cada enfoque?

Con la evidencia del 4.4.A, 4.4.B y la síntesis de la sección 3, podemos cerrar el 4.4 con una **guía decisional** que destila lo aprendido en pocas reglas.

El árbol de abajo se lee de arriba hacia abajo: comenzás por el **tipo de problema** que tenés (regresión tabular o serie temporal), seguís por una pregunta de diagnóstico (no-linealidades sospechosas, tamaño del dataset), y llegás a un modelo recomendado.

```
                          ┌──────────────────────────────┐
                          │     Tipo de problema         │
                          └──────────────┬───────────────┘
                                         │
                ┌────────────────────────┴────────────────────────┐
                │                                                 │
        ┌───────▼────────┐                                ┌───────▼────────┐
        │  Regresión     │                                │  Series        │
        │  tabular       │                                │  temporales    │
        └───────┬────────┘                                └───────┬────────┘
                │                                                 │
        ¿No-linealidades                                  ¿Cuántos datos?
        sospechosas?                                       │
         │              │                                  │              │
       NO ▼            SÍ ▼                          <200 ▼          ≥1000 ▼
    ┌──────────┐  ┌──────────┐                   ┌─────────────┐   ┌──────────────┐
    │ Ridge /  │  │   MLP    │                   │ ARIMA o     │   │ LSTM tuneada │
    │  Lasso   │  │          │                   │ LSTM simple │   │              │
    └──────────┘  └──────────┘                   └─────────────┘   └──────────────┘
    Rápido,       Captura no-                    Margen modesto,    DL gana fuerte
    interpretable lin. implícitas                tuning poco útil

                           ┌─────────────────────────────────────────┐
                           │  Regla transversal:                      │
                           │  Si necesitás explicar el modelo a       │
                           │  un comité regulatorio, los coeficientes │
                           │  lineales te salvan; un MLP o LSTM no.   │
                           └─────────────────────────────────────────┘
```

### 4.1 Las cuatro reglas, escritas con palabras

**Regla 1: Si tenés regresión tabular sin sospecha de no-linealidades → empezá con Ridge o Lasso.**  
Son rápidos, interpretables, y si el problema no esconde transformaciones no lineales, no vas a perder casi nada con respecto a un MLP. La regla práctica que vimos en el 4.1: si pensás que hay un cociente o un producto entre features que importa, calculalo a mano y agregalo como columna. Ridge con `energy_per_capita` agregada manualmente alcanzó la performance del MLP.

**Regla 2: Si tenés regresión tabular y dudás si hay no-linealidades → tirá un MLP exploratorio.**  
Comparalo contra Ridge. Si el MLP gana fuerte (como en nuestro 4.4.A: 0.93 vs 0.76), hay algo no-lineal que vale la pena descubrir —usá SHAP o partial dependence plots para identificar la transformación oculta y, si podés, codificala como feature manual—. Si gana poco, no vale el costo de mantenimiento.

**Regla 3: Si tenés serie temporal con menos de 200 puntos → ARIMA o LSTM Simple, sin tunear.**  
Con datasets chicos el techo de mejora es bajísimo y la varianza inter-seed se come el margen. Nuestro 4.4.B mostró que la LSTM Tuneada empata a la Simple dentro del ruido (±3.47 en MAPE). Tunear con poco datos es teatro: produce números variables y poco transferibles.

**Regla 4: Si tenés serie temporal con muchos datos → ahí sí, Deep Learning brilla.**  
No tenemos esta evidencia en el proyecto (nuestra serie es chica), pero la literatura es consistente: con >1000 puntos las LSTMs y los Transformers temporales superan claramente a ARIMA. La regla del pulgar es simple: las redes necesitan datos para aprender la dinámica, y nuestra serie de 96 meses estaba en la zona donde el costo extra del DL apenas paga.

**Regla transversal: la interpretabilidad no es negociable en algunos contextos.**  
Si tu modelo va a alimentar una decisión que un regulador, un comité de ética o un cliente exigente puede pedir explicar, los modelos lineales son tu salvación. Un MLP con 10.000 parámetros y un LSTM con 4.000 son cajas negras —incluso con SHAP encima, la explicación es post-hoc, no estructural—. Esto es importante en finanzas, salud, política pública, o cualquier dominio con auditoría.

---

## 5. Storytelling de cierre del Paso 4.4

Si el 4.4 fuera una sola enseñanza, sería esta: **"Deep Learning gana" no es una regla, es una hipótesis a probar caso por caso.** Lo que demostramos en este paso es que el match entre modelo y problema importa más que la sofisticación del modelo en sí mismo.

Las grandes lecciones, en orden:

### 1️⃣ El Deep Learning ganó cuando había no-linealidades para descubrir, perdió cuando no

En el bloque A —regresión global de CO₂ per cápita— el MLP barrió a los lineales por **17 puntos de R²**. Pero lo que ganó no fue "capacidad analítica": fue la habilidad de descubrir implícitamente la transformación `energy_per_capita = energy_Mtoe / poblacion`. Cuando se la dimos a Ridge como feature manual, Ridge igualó al MLP. **El MLP no era más inteligente: era mejor buscando cocientes ocultos.**

En el bloque B —forecasting univariado— no había features para combinar. Lo único que la LSTM podía aprender era la dinámica temporal, y eso ARIMA también lo intenta. Resultado: la LSTM le ganó a ARIMA, pero por márgenes modestos (~6 pts de MAPE), y el tuning no aportó mejora robusta.

### 2️⃣ La cantidad de datos manda más que la elección del modelo

El bloque A tenía 458 filas; el bloque B, 72 ejemplos de entrenamiento. Esa diferencia de **6×** explica gran parte del comportamiento dispar:

| | Bloque A (n=458) | Bloque B (n=72) |
|---|---|---|
| Margen DL vs clásico | ~17 pts R² | ~6 pts MAPE |
| Varianza inter-seed | ±0.018 (R²) | ±3.47 (MAPE) |
| Margen / Ruido | ~95× | ~2× |
| ¿Tuning ayuda? | (no testeado) | No, dentro del ruido |

Con muchos datos, la jerarquía DL > clásico es nítida. Con pocos datos, se difumina hasta volverse estadísticamente cuestionable.

### 3️⃣ Reportar una sola corrida es académicamente cuestionable

La diferencia entre la mejor y la peor seed de la LSTM Tuneada fue de **6.87 puntos de MAPE** (17.55% vs 24.42%) — más grande que la diferencia entre LSTM y ARIMA. Si hubiéramos reportado solo la mejor seed, habríamos vendido la Tuneada como un avance espectacular. Si hubiéramos reportado solo la peor, como un fracaso. La verdad estaba en el medio, y solo se ve con 3 seeds.

Esta es probablemente **la lección práctica más importante** del proyecto para un futuro empleador: cualquier resultado de red neuronal con dataset chico que no reporte varianza inter-seed merece desconfianza por defecto.

### 4️⃣ La interpretabilidad sigue importando

Los modelos lineales que la academia trata como "baseline" son, en muchos contextos del mundo real, los modelos preferidos —no por performance, sino por explicabilidad—. Una regulación financiera, un sistema de scoring crediticio, o una decisión de política pública requieren que el modelo se pueda auditar. Un coeficiente de Ridge se audita en 30 segundos; un MLP, no.

### 5️⃣ Hay un trade-off real entre automatización y comprensión

El MLP del 4.1 hizo feature engineering por nosotros: encontró `energy_per_capita`. Eso es valioso cuando no sabés qué buscar y caro cuando sabés exactamente qué buscás. La elección entre clásico y DL muchas veces no es "qué predice mejor" sino "cuánto del trabajo querés delegar al modelo". En equipos chicos, automatizar feature engineering vale oro; en equipos grandes con expertise de dominio, los lineales con buenas features compiten cabeza a cabeza.

---

## 6. Próximo paso

Con el Paso 4.4 cerrado, queda un solo paso en la Etapa 4:

**Paso 4.5 — Storytelling final del proyecto completo.** Una narrativa coherente que conecte la EDA (Etapa 1), la limpieza de datos (Etapa 2), los modelos clásicos (Etapa 3) y la Etapa 4 entera. El objetivo es entregar un documento ejecutivo que un lector no-técnico pueda recorrer de principio a fin y entender qué aprendimos sobre cambio climático y energías renovables en Argentina.

*Notebook ejecutado para el Desafío Profesional de Data Science — Digital House.*